##AIによる国会議事録のQA抽出 ハンズオン（Section 09）

このノートブックでは、OpenAI APIを使って国会議事録の発言データから
質問と回答を生成し、データベースへ保存する流れを体験します。

----

ここでやること

- OpenAI API を利用する準備をする
- API呼び出し用の共通関数を作成する
- ank.db をダウンロードして内容を確認する
- 発言データを読み込み、AIでQAを生成する
- 生成したQAをJSON形式で受け取る
- qa テーブルへ登録し、保存結果を確認する
- 次の章で使うナレッジデータを作る

In [ ]:
# Colabで config.py を作る（同じ作業ディレクトリに作成）
api_key = "XXXXXXXXXX"  # ←自分のキーに置き換え

with open("config.py", "w", encoding="utf-8") as f:
    f.write(f'OPENAI_API_KEY = "{api_key}"\n')

print("config.py を作成しました")

In [ ]:
from openai import OpenAI
import config

# OpenAIクライアント作成
client = OpenAI(api_key=config.OPENAI_API_KEY)

# オウム返しのベースとなる入力
user_text = "こんにちは、これは安いモデルでのテストです。"

# API呼び出し（最安モデル）
response = client.responses.create(
    model="gpt-4o-mini",   # ❗ 最も安価なモデル
    input=f"次の文章をそのまま繰り返してください：{user_text}"
)

# 結果出力
print(response.output_text)

In [ ]:
ank_requests_code = r'''
from openai import OpenAI
import config

def send_request(prompt: str, model: str = "gpt-4o-mini") -> str:
    """
    OpenAI API に prompt を送信し、
    返ってきたテキストをそのまま返す

    Parameters
    ----------
    prompt : str
        モデルに渡す入力文
    model : str
        使用するモデル名（省略時は gpt-4o-mini）
    """
    client = OpenAI(api_key=config.OPENAI_API_KEY)

    response = client.responses.create(
        model=model,
        input=prompt
    )

    return response.output_text
'''

with open("ank_requests.py", "w", encoding="utf-8") as f:
    f.write(ank_requests_code)

print("ank_requests.py を作成しました")

In [ ]:
from openai import OpenAI
import ank_requests

# オウム返しのベースとなる入力
user_text = "次の文章をそのまま繰り返してください：こんにちは、これは安いモデルでのテストです。"

# 関数にプロンプトを渡す
answer = ank_requests.send_request(user_text)

# 結果出力
print(answer)

In [ ]:
import requests

print("Githubからのダウンロードを開始します")

url = "https://raw.githubusercontent.com/aiknowledgelearning-decuments/document01/main/section09/ank.db"
r = requests.get(url)

with open("ank.db", "wb") as f:
    f.write(r.content)

print("Githubからのダウンロードが完了しました")


In [ ]:
import sqlite3
import pandas as pd

# GitHubなどからダウンロードした ank.db を指定
db_file = "ank.db"

# 接続
conn = sqlite3.connect(db_file)

# テーブル一覧を取得
tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
AND name NOT LIKE 'sqlite_%'
ORDER BY name
""", conn)

# 結果格納
result = []

# 各テーブルの件数確認
for table_name in tables["name"]:
    cnt = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {table_name}", conn)
    result.append({
        "テーブル名": table_name,
        "件数": cnt.loc[0, "cnt"]
    })

# DataFrame化
df_result = pd.DataFrame(result)

# 表示
display(df_result)

# 切断
conn.close()

print("確認が完了しました")

In [ ]:
# 発言からＱＡを生成
import sqlite3
import pandas as pd
import ank_requests

DB_FILE = "ank.db"
CHUNK_SIZE = 5   # 実行する発言件数

conn = sqlite3.connect(DB_FILE)

df_speeches = pd.read_sql("""
SELECT speechID, speaker, speech FROM speeches ORDER BY SpeechID
""", conn)

conn.close()

print("取得件数:", len(df_speeches))
print()

# -------------------------
# 先頭から CHUNK_SIZE 件だけ処理
# -------------------------
for i, (_, row) in enumerate(df_speeches.iterrows(), start=1):

    if i > CHUNK_SIZE:
        break

    speech_text = row["speech"]

    prompt = f"""
あなたは優秀な編集者です。

以下の国会議事録の発言内容を読み取り、
重要な内容を質問と回答の形で5件作成してください。

【条件】
・質問は分かりやすくする
・回答は本文に沿って簡潔にする
・Q1. A1. 形式で出力する
・発言にない内容は追加しない

【発言内容】
{speech_text}
"""

    print("=" * 80)
    print(f"{i}件目 / speechID = {row['speechID']}")
    print(f"発言者: {row['speaker']}")
    print("-" * 80)

    answer = ank_requests.send_request(prompt)

    print(answer)
    print()

In [ ]:
# 生成したＱＡをＱＡテーブルに保存
import sqlite3
import pandas as pd
import ank_requests
import json
import re

DB_FILE = "ank.db"
CHUNK_SIZE = 5   # 実行する発言件数

conn = sqlite3.connect(DB_FILE)

df_speeches = pd.read_sql("""
SELECT speechID, speaker, speech FROM speeches ORDER BY speechID
""", conn)

print("取得件数:", len(df_speeches))
print()

for i, (_, row) in enumerate(df_speeches.iterrows(), start=1):

    if i > CHUNK_SIZE:
        break

    speech_id = row["speechID"]
    speech_text = row["speech"]

    prompt = f"""
あなたは優秀な編集者です。

以下の国会議事録の発言内容を読み取り、
重要な内容を質問と回答の形で5件作成してください。

出力は必ずJSONのみとしてください。
説明文、前置き、補足は不要です。

出力形式は以下の配列形式にしてください。

[
  {{
    "question": "質問文",
    "answer": "回答文"
  }},
  {{
    "question": "質問文",
    "answer": "回答文"
  }},
  {{
    "question": "質問文",
    "answer": "回答文"
  }},
  {{
    "question": "質問文",
    "answer": "回答文"
  }},
  {{
    "question": "質問文",
    "answer": "回答文"
  }}
]

条件:
- 質問は分かりやすくする
- 回答は本文に沿って簡潔にする
- 発言にない内容は追加しない

発言内容:
{speech_text}
"""

    print("=" * 80)
    print(f"{i}件目 / speechID = {speech_id}")
    print(f"発言者: {row['speaker']}")
    print("-" * 80)

    answer = ank_requests.send_request(prompt)

    try:
        answer = re.sub(r"^```json\s*", "", answer.strip())
        answer = re.sub(r"\s*```$", "", answer.strip())
        qa_list = json.loads(answer)
    except Exception as e:
        print("JSON変換エラー")
        print(answer)
        print(e)
        continue

    conn.execute("DELETE FROM qa WHERE speechID = ?", (speech_id,))

    for qa in qa_list:
        question = qa.get("question", "").strip()
        answer_text = qa.get("answer", "").strip()

        if not question or not answer_text:
            continue

        conn.execute("""
        INSERT INTO qa (speechID, question, answer)
        VALUES (?, ?, ?)
        """, (speech_id, question, answer_text))

    conn.commit()

    print(f"qaテーブルへ {len(qa_list)} 件登録しました")
    print()

conn.close()

print("処理完了")